`r format(Sys.time(), '🗓️ %d %B, %Y 🕔 %H:%M')`

In [ ]:
library(BioCircos)
library(dplyr)
library(DT)
library(tidyr)
library(tools)
library(viridisLite)

In [ ]:
indir <- params$statsdir
statsfiles <- list.files(path = indir, pattern = "*sv.stats", full.names = T)
faidx <- params$faidx

In [ ]:
readvariants <- function(x){
  read.table(
    x,
    header = T,
    colClasses = c("factor", "factor", "integer", "integer", "character", "factor", "integer", "integer")
  )
}

# General Stats

In [ ]:
emptyfiles <- character()
sv <- data.frame(matrix(ncol=8,nrow=0, dimnames=list(NULL, c("population",	"contig",	"position_start",	"position_end",	"length",	"type",	"n_barcodes",	"n_pairs"))))
for(i in statsfiles){
  .df <- tryCatch(readvariants(i), error = function(e){return(0)})
  if(nrow(.df)==0){
    emptyfiles <- c(emptyfiles, gsub(".bedpe", "", basename(i)))
  } else {
    sv <- rbind(sv, .df)
  }
}
if(length(emptyfiles) == length(statsfiles)){
  cat("There are no variants detected in any of the populations:\n")
  cat(paste0("- ", statsfiles), sep = "\n")
  knitr::knit_exit()
}
if(length(emptyfiles > 0)){
  cat("Some of the populations did not have any variants detected and are omitted from this report:\n")
  cat(paste0("- ", emptyfiles), sep = "\n")
}

In [ ]:
summinfo <- sv %>%
  group_by(population, type) %>%
  summarise(tot = n()) %>%
  group_by(type) %>%
  summarise(count = sum(tot), avg = round(mean(tot, na.rm = T),2 ), sd = round(sd(tot, na.rm = T), 2)) #%>% 

##

In [ ]:
list(
  color = "light",
  value = prettyNum(length(levels(sv$population)), big.mark = ",")
)

In [ ]:
nbnd <- ifelse("BND" %in% summinfo$type, summinfo$avg[which(summinfo$type == "BND")], 0)
list(
  color = "#dfdfdf",
  value = prettyNum(nbnd, big.mark = ",")
)

In [ ]:
ndel <- ifelse("DEL" %in% summinfo$type, summinfo$avg[which(summinfo$type == "DEL")], 0)
list(
  color = "#dfdfdf",
  value = prettyNum(round(ndel,1), big.mark = ",")
)

In [ ]:
ndup <- ifelse("DUP" %in% summinfo$type, summinfo$avg[which(summinfo$type == "DUP")], 0)
list(
  color = "#dfdfdf",
  value = prettyNum(round(ndup,1), big.mark = ",")
)

In [ ]:
ninv <- ifelse("INV" %in% summinfo$type, summinfo$avg[which(summinfo$type == "INV")], 0)
list(
  color = "#dfdfdf",
  value = prettyNum(round(ninv,1), big.mark = ",")
)

##
This report details the structural variants identified by [LEVIATHAN](https://github.com/morispi/LEVIATHAN)
[(preprint)](https://doi.org/10.1101/2021.03.25.437002), which was run in sample-pooling mode. This tab (`General Stats`)
shows overview information. The `Per-Contig Plot` tab shows you an interactive
plot detailing all variants identified. The variants are given by:

INV (Inversion)
: A segment that broke off and reattached within the same chromosome, but in reverse orientation

DUP (Duplication)
: A type of mutation that involves the production of one or more copies of a gene or region of a chromosome

DEL (Deletion)
: A type of mutation that involves the loss of one or more nucleotides from a segment of DNA

BND (Breakend)
: The endpoint of a structural variant, which can be useful to explain complex variants

## Various information
::: {.card title="All variants found" expandable="true"}
This table details all the variants LEVIATHAN detected and evidence for those detections.

In [ ]:
sv$length <- as.numeric(gsub("\\.", "1", sv$length))

In [ ]:
DT::datatable(
  sv,
  rownames = F,
  filter = "top",
  extensions = 'Buttons',
  fillContainer = T,
  options = list(
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = "leviathan_sv")),
    scrollX = T,
    paging = T
  )
)

:::

::: {.card title="Variants per Contig" expandable="true"}
This table details counts each type of structural variant for every contig for every population.

In [ ]:
grpcounts <- sv %>%
  group_by(population, contig, type) %>%
  summarise(count = n()) %>%
  pivot_wider(names_from = population, values_from = count)

DT::datatable(
  grpcounts,
  rownames = F,
  filter = "top",
  extensions = 'Buttons',
  fillContainer = T,
  options = list(
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = "leviathan_sv_count_pop")),
    scrollX = T,
    paging = T
  )
)

:::

## bp per contig
::: {.card title="Variants by bp"}

This table details the lengths (in bp) of structural variants for every contig for every population. Breakends are not counted because they do not have a size.

In [ ]:
grplens <- sv %>% 
  filter(type != "BND") %>%
  group_by(population, contig, type) %>%
  summarise(bp = sum(length)) %>%
  pivot_wider(names_from = population, values_from = bp)

DT::datatable(
  grplens,
  rownames = F,
  filter = "top",
  extensions = 'Buttons',
  fillContainer = T,
  options = list(
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = "leviathan_sv_bp_pop")),
    scrollX = T,
    paging = T
  )
)

:::

# Inversions

In [ ]:
fa.sizes <- read.table(faidx, header = F)[,1:2] %>% arrange(desc(2))
colnames(fa.sizes) <- c("contig", "size")
plot_contigs <- params$contigs
if (all(plot_contigs == "default")){
  # make sure that the contigs with SV are the ones being plotted, not the others
  fa.sizes <- fa.sizes[fa.sizes$contig %in% unique(sv$contig), ]
  # limit the data to only the 30 of the largest present contigs
  fa.sizes <- fa.sizes[1:min(nrow(fa.sizes), 30), ]
} else {
  fa.sizes <- fa.sizes[fa.sizes$contig %in% plot_contigs, ]
}

# filter out variants that aren't on one of these contigs
sv <- sv[sv$contig %in% fa.sizes$contig, ]
sv$contig <- factor(sv$contig)

populations <- levels(sv$population)
color.palette <- turbo(length(populations))
names(color.palette) <- populations

genomeChr <- fa.sizes$size
names(genomeChr) <- fa.sizes$contig
genomeChr <- as.list(genomeChr)
radius <- 0.9
radius_increments <- round(.6 / length(populations), 3)
padding <- radius_increments * 0.2

In [ ]:
calculate_luminance <- function(hex) {
  rgb_values <- col2rgb(hex) / 255
  luminance <- 0.2126 * rgb_values[1] + 0.7152 * rgb_values[2] + 0.0722 * rgb_values[3]
  return(luminance)
}

# Function to decide text color based on background luminance
get_text_color <- function(background_color) {
  luminance <- calculate_luminance(background_color)
  if (luminance > 0.5) {
    # Dark background -> Light text (black)
    return("black") 
  } else {
    # Light background -> Dark text (white)
    return("white") 
  }
}
poplegend <- DT::datatable(
  data.frame(Group = populations),
  rownames = F,
  fillContainer = T,
  options = list(
    dom = 't',
    scrollX = F,
    scrollY = F,
    paging = F
  ),
) %>% formatStyle(
  "Group",
  backgroundColor = styleEqual(names(color.palette), color.palette),
  # Adjust text color dynamically
  color = styleEqual(names(color.palette), sapply(color.palette, get_text_color))
)

## {.fit}
<h2>Detected Inversions</h2>

This is a circular plot to visualize the distribution of _putative_ inversions across (up to) 30 of the largest contigs
(or the ones that were specified) across the groups you specified in your data.
If you are unfamiliar with this kind of visualization, it's a circular representation of a linear genome.
Each "wedge" is a different contig, from position 0 to  the end of the contig, and is labelled by the contig name.
Each internal (grey) ring is detected inversions on that contig for a different population. Very small variants
may be difficult to see or will not appear on the plot.

##
::: {.card width="15%" title="Legend"}

In [ ]:
poplegend

:::
###

In [ ]:
radius <- 0.9
tracks <- BioCircosTracklist()
variant_type <- "INV"
# Add one track for each population
for( pop in populations ){
  variant <- sv[(sv$type == variant_type) & (sv$population == pop),]
  if(nrow(variant) > 0){
    tracks <- tracks + BioCircosArcTrack(
      paste0(variant_type, pop),
      as.character(variant$contig),
      variant$position_start,
      variant$position_end,
      opacities = 0.7,
      values = variant$n_barcodes,
      minRadius = radius - (radius_increments - padding),
      maxRadius = radius - padding,
      colors = color.palette[pop],
      labels = paste0("Population: ", pop, "</br>Type: Inversion"),
    )
  }
  # loop to create backgrounds
  tracks <- tracks + BioCircosBackgroundTrack(
    paste0(pop,"_background"),
    minRadius = radius - radius_increments, maxRadius = radius,
    fillColors = "#F9F9F9",
    borderColors = "#C9C9C9",
    borderSize = 1
  )
  radius <- radius - radius_increments
}
BioCircos(
  tracks,
  displayGenomeBorder = F,
  genome = genomeChr,
  chrPad = 0.03,
  genomeTicksDisplay = F,
  genomeLabelDy = 3,
  genomeLabelTextSize = 20,
  SNPMouseOverCircleSize = 2,
  SNPMouseOverTooltipsHtml03 = "<br/>Length: ",
  width = "100%",
  height = "1000px",
  elementId = paste0(variant_type, "_circosplot") 
)

# Duplications
## {.fit}
<h2>Detected Duplications</h2>

This is a circular plot to visualize the distribution of _putative_ duplications across (up to) 30 of the largest contigs
(or the ones that were specified) across the groups you specified in your data.
If you are unfamiliar with this kind of visualization, it's a circular representation of a linear genome.
Each "wedge" is a different contig, from position 0 to  the end of the contig, and is labelled by the contig name.
Each internal (grey) ring is the detected duplications on that contig for a different population. Very
small variants may be difficult to see or will not appear on the plot.

##
::: {.card width="15%" title="Legend"}

In [ ]:
poplegend

:::

###

In [ ]:
# Add one track for each population
radius <- 0.9
tracks <- BioCircosTracklist()
variant_type <- "DUP"
# Add one track for each population
for( pop in populations ){
  variant <- sv[(sv$type == variant_type) & (sv$population == pop),]
  if(nrow(variant) > 0){
    tracks <- tracks + BioCircosArcTrack(
      paste0(variant_type, pop),
      as.character(variant$contig),
      variant$position_start,
      variant$position_end,
      opacities = 0.7,
      values = variant$n_barcodes,
      minRadius = radius - (radius_increments - padding),
      maxRadius = radius - padding,
      colors = color.palette[pop],
      labels = paste0("Population: ", pop, "</br>Type: Duplication"),
    )
  }
  # loop to create backgrounds
  tracks <- tracks + BioCircosBackgroundTrack(
    paste0(pop,"_background"),
    minRadius = radius - radius_increments, maxRadius = radius,
    fillColors = "#F9F9F9",
    borderColors = "#C9C9C9",
    borderSize = 1
  )
  radius <- radius - radius_increments
}

BioCircos(
  tracks,
  displayGenomeBorder = F,
  genome = genomeChr,
  chrPad = 0.03,
  genomeTicksDisplay = F,
  genomeLabelDy = 3,
  genomeLabelTextSize = 20,
  width = "100%",
  height = "1000px",
  elementId = paste0(variant_type, "_circosplot") 
)

# Deletions
## {.fit}
<h2>Detected Deletions</h2>

This is a circular plot to visualize the distribution of _putative_ deletions across (up to) 30 of the largest contigs
(or the ones that were specified) across the groups you specified in your data.
If you are unfamiliar with this kind of visualization, it's a circular representation of a linear genome.
Each "wedge" is a different contig, from position 0 to  the end of the contig, and is labelled by the contig name.
Each internal (grey) ring is the detected deletions for that contig for a different population. Very small variants
may be difficult to see or will not appear on the plot. Deletions are displayed as points as they tend to be small
and clump together.

##
::: {.card width="15%" title="Legend"}

In [ ]:
poplegend

:::
###

In [ ]:
# Add one track for each population
radius <- 0.9
tracks <- BioCircosTracklist()
variant_type <- "DEL"
# Add one track for each population
for( pop in populations ){
  variant <- sv[(sv$type == variant_type) & (sv$population == pop),]
  if(nrow(variant) > 0){
    tracks <- tracks + BioCircosSNPTrack(
      paste0(variant_type, pop),
      as.character(variant$contig),
      variant$position_start,
      values = abs(variant$position_end - variant$position_start),
      minRadius = radius - (radius_increments - padding),
      maxRadius = radius - padding,
      size = 2,
      shape = "rect",
      opacities = 0.7,
      colors = color.palette[pop],
      labels = paste0("Population: ", pop, "</br>Type: Deletion"),
  )
  }
  # loop to create backgrounds
  tracks <- tracks + BioCircosBackgroundTrack(
    paste0(pop,"_background"),
    minRadius = radius - radius_increments, maxRadius = radius,
    fillColors = "#F9F9F9",
    borderColors = "#C9C9C9",
    borderSize = 1
  )
  radius <- radius - radius_increments
}

BioCircos(
  tracks,
  displayGenomeBorder = F,
  genome = genomeChr,
  chrPad = 0.03,
  genomeTicksDisplay = F,
  genomeLabelDy = 3,
  genomeLabelTextSize = 20,
  SNPMouseOverCircleSize = 2,
  SNPMouseOverTooltipsHtml03 = "<br/>Length: ",
  width = "100%",
  height = "1000px",
  elementId = paste0(variant_type, "_circosplot") 
)